# Optimization Campaign (HITL)

**Prerequisites:** TermNorm backend at `http://127.0.0.1:8000` | Groq API key in `.env` | Restart kernel after first sync

**Workflow:** Setup → Data → Explore → Optimize → Results

In [ ]:
%load_ext autoreload
%autoreload 2

import json
from _campaign_lib import *

# --- Services ---
svc = init_services()
TASK_DESCRIPTION = load_task_description(
    r"C:\Users\dsacc\OfficeAddinApps\TermNorm-excel\backend-api\config\LCA_INPUT_PATTERNS.md"
)

# --- Campaign config ---
campaign_config = {
    "sample_size": 15,              # queries per eval step (service default: all)
    "exploration_rate": 0.5,             # PRIMARY KNOB: 0.0=conservative, 1.0=aggressive
    "improvement_areas": "profile schema quality, web search relevance",
    "exclude_steps": ["llm_ranking"],    # steps to skip (e.g. ["entity_profiling"])
    "pipeline_overrides": {},
    "optimization": {
        "patience": 2,                   # default: 3
        "max_rounds": None,              # default: 10 (None = unlimited, patience-only stop)
        "degradation_threshold": 0.4,    # fraction of degraded queries to trigger escalation (0 = disabled)
        "enable_l2": True,               # L2 refine_context on escalation
        "enable_l3": True,               # L3 modify_plan on L2 stall
        "l2_patience": None,             # default: 2 (None = unlimited L2 rounds)
        "l3_patience": None,             # default: 1 (None = unlimited L3 rounds)
    },
    "eval_llm": {
        # --- Groq (free tier, open-source models) ---
        "model": "openai/gpt-oss-120b",
        # "model": "moonshotai/kimi-k2-instruct-0905"
        "provider_url": "https://api.groq.com/openai/v1/chat/completions",
        # --- Anthropic (cost: opus >> sonnet >> haiku) ---
        # "model": "claude-opus-4-6",          # best quality
        # "model": "claude-sonnet-4-6",      # good balance
        # "model": "claude-haiku-4-5-20251001",  # cheapest
        # "provider_url": "https://api.anthropic.com",
        "max_tokens": 2000,              # response length budget
    },
    "grid_search": {
        "context": "A terminology normalization pipeline that matches raw material "
                    "descriptions to standardized database terms using entity profiling "
                    "and candidate ranking.",
        "grid_budget": 35,               # default: 0 (full grid)
        "sample_size": 6,     # default: 0 (all queries)
        "shared_queries": False,          # default: True
    },
}

# --- Pipeline snapshot & params ---
pipeline_config_full = show_pipeline_snapshot(svc)
pipeline_params = configure_pipeline(svc, campaign_config)

# --- Data ---
EXCEL_PATH = r"C:\Users\dsacc\Desktop\project-TermNorm\OneDrive_2025-07-02\Austausch Beispiele\Prozessnamen\BOM-example.xlsx"  # e.g. "../data/BOM-example.xlsx"
FORCE_RELOAD = False  # Set True to re-read Excel and overwrite stored datasets

train_data, svc["session_terms"] = prepare_datasets(
    svc["store"], svc["backend_id"],
    excel_path=EXCEL_PATH or None,
    force=FORCE_RELOAD,
)

In [ ]:
#@title Task context decomposition
task_context = await decompose_task_context(TASK_DESCRIPTION, campaign_config, svc)

In [ ]:
#@title Prepare evaluation context
campaign_rounds = []
baseline_results = []

baseline_ps, eval_data, backend_status = prepare_eval_context(
    svc, train_data,
)

RUN_BASELINE = False  # Set True to evaluate baseline before exploration
if RUN_BASELINE:
    campaign_rounds, baseline_results = run_baseline_eval(
        baseline_ps, eval_data, campaign_config, svc,
    )

In [6]:
#@title Experiment dashboard
# Set to a short hex ID (e.g. '68e2c5') to resume a specific experiment.
# The system adds prefixes (cycle_, scan_, etc.) per data type.
# Set to None to auto-detect from current campaign_config + eval_data.
EXPERIMENT_ID = '68e2c53845c3' #None

# When EXPERIMENT_ID is set, load stored config → overrides notebook variables
if EXPERIMENT_ID:
    stored_cfg = load_experiment_config(svc["store"], svc["backend_id"], EXPERIMENT_ID)
    if stored_cfg:
        pp_override = apply_experiment_overrides(campaign_config, stored_cfg)
        if pp_override:
            pipeline_params = pp_override
        print(f"  Loaded config from experiment {EXPERIMENT_ID}")

show_experiment_dashboard(
    svc=svc, experiment_id=EXPERIMENT_ID,
    campaign_config=campaign_config, eval_data=eval_data,
    pipeline_params=locals().get("pipeline_params"),
    baseline_prompt_state=campaign_rounds[0]["prompt_state"].model_dump() if campaign_rounds else None,
)

  Loaded config from experiment 68e2c53845c3

  EXPERIMENT: cycle_68e2c53845c3
  Status: interrupted  |  Rounds: 2  |  Best: 20.0%  |  Base: 20.0%
  Updated: 2026-03-19 09:52

  Config (copy to campaign_config to resume):
    max_rounds: 3
    patience: 2
    n_variants: 5
    creativity: 0.7
    improvement_threshold: 0.01
    model: openai/gpt-oss-120b
    temperature: 0.0
    sample_size: 15
    seed: 42
    pipeline_params: {max_token_candidates=30, profiling_schema=..., profiling_temperature=0.3, query_prefix=what material is, steps=...}


Diff: current config vs cycle_68e2c53845c3
  (identical — will resume this campaign)
  → Config does NOT match — update campaign_config to resume



{'campaign_id': 'cycle_68e2c53845c3',
 'created_at': '2026-03-12T16:07:56.174046+00:00',
 'updated_at': '2026-03-19T09:52:43.808141+00:00',
 'status': 'interrupted',
 'n_trials': 2,
 'best_accuracy': 0.2,
 'best_trial_id': 'round_0',
 'baseline_accuracy': 0.2,
 'trials': [{'trial_id': 'round_1',
   'round': 1,
   'label': 'round_1',
   'prompt_state_id': '',
   'accuracy': 0.2,
   'hits': 0,
   'total': 0,
   'improved': False,
   'created_at': ''},
  {'trial_id': 'round_0',
   'round': 0,
   'label': 'baseline',
   'prompt_state_id': '',
   'accuracy': 0.2,
   'hits': 0,
   'total': 0,
   'improved': False,
   'created_at': ''}],
 'type': 'feedback_cycle',
 'config': {'max_rounds': 3,
  'patience': 2,
  'n_variants': 5,
  'creativity': 0.7,
  'improvement_threshold': 0.01,
  'model': 'openai/gpt-oss-120b',
  'provider': None,
  'backend_url': 'http://127.0.0.1:8000',
  'backend_id': 'termnorm-local',
  'project_root': 'C:\\Users\\dsacc\\Desktop\\PromptPotter\\prompt-potter-optimizer\\

## 3. Explore

Two exploration paths: **Smart Search** (scan advisor + sensitivity scan) or **Grid Search** (brute-force sweep). Use one or both.

In [7]:
#@title 3a. Smart Search — Browse variant library
# display_variant_library()
# Filter examples:
# display_variant_library(source="PromptWizard")
# display_variant_library(axes=["thinking_style", "persona"])

In [ ]:
# preview_advisor_prompt()
preview_advisor_prompt(campaign_config, svc, task_description=task_context, raw=True)

In [ ]:
#@title Scan advisor
advisory, scan_variants, schema_labels = await run_scan_advisor(
    campaign_config, svc,
    task_description=locals().get("task_context") or locals().get("TASK_DESCRIPTION", ""),
)

In [11]:
#@title Scan variant config (edit suggested values or add your own)
# Schema axes: mutation tuples ("-", path), ("+", path, type, req, desc),
# ("~", old, new, type, req, desc). Non-schema axes: plain value lists.

scan_sample_size = 10  # queries per scan variant (0 = use all)

scan_variants = {
    'max_token_candidates': [10, 30, 50],
    'query_prefix': ['what material is', 'identify LCA database name for', 'translate trade name'],
    'profiling_schema': [
        [['+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA'], ['+', 'database_format_hint', 'string', False, "Best-guess ecoinvent-style name fragment for this entity, e.g. 'market for polyethylene, high density'"]],
        # [['-', 'manufacturing_processes'], ['-', 'applications'], ['+', 'lca_synonyms', 'array', False, 'Terms likely to appear verbatim in LCA database entry names for this entity'], ['+', 'no_match_signal', 'string', False, 'Brief reasoning on whether a database match is likely to exist or not']],
        # [['~', 'classification_aliases', 'lca_classification_aliases', 'array', False, 'Expert-level aliases specifically aligned with LCA database naming conventions, including ecoinvent activity names and SimaPro process names'], ['+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA']],
        [['+', 'lca_database_names', 'array', True, "Likely ecoinvent or GaBi database entry names that would match this entity, using standard LCA database naming conventions like 'market for X | X | cut-off, U'"]], 
        [['-', 'manufacturing_processes'], ['-', 'applications'], ['+', 'lca_database_names', 'array', True, 'Likely ecoinvent or GaBi database entry names for this entity using standard LCA naming conventions']],
        [['~', 'notes', 'material_category', 'string', True, "The broad LCA material category this entity belongs to, e.g. 'polyethylene', 'brass', 'steel'"]]
    ],
    'profiling_temperature': [0.0, 0.3, 0.7],
    # 'profiling_max_tokens': [512, 1024, 2048], # -> Going to cause lots of Errors.
    'raw_content_limit': [1000, 2500, 8000],
}
scan_variants, schema_labels = resolve_scan_variants(scan_variants, svc=svc)

  max_token_candidates: [10, 30, 50]
  query_prefix: ['what material is', 'identify LCA database name for', 'translate trade name']
  profiling_schema: (baseline + 4 mutations)
    [0] (baseline)
    [1] ('+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA'), ('+', 'database_format_hint', 'string', False, 'Best-guess ecoinvent-style name fragment for this entity, e.g. 'market for polyethylene, high density'')
    [2] ('+', 'lca_database_names', 'array', True, 'Likely ecoinvent or GaBi database entry names that would match this entity, using standard LCA database naming conventions like 'market for X | X | cut-off, U'')
    [3] ('-', 'manufacturing_processes'), ('-', 'applications'), ('+', 'lca_database_names', 'array', True, 'Likely ecoinvent or GaBi database entry names for this entity using standard LCA naming conventions')
    [4] ('~', 'notes', 'material_category', 'string', True, 'The broad LCA material 

In [12]:
#@title Prepare scan baseline
# Scan always uses fresh pipeline defaults (not experiment overrides) so the
# baseline content hash matches previous runs regardless of EXPERIMENT_ID.
scan_pipeline_params = configure_pipeline(svc, campaign_config)
scan_baseline_sp, scan_coverage = await prepare_scan_baseline(
    baseline_ps, campaign_config,
    pipeline_params=scan_pipeline_params,
    svc=svc, scan_variants=scan_variants,
)

2026-03-19 10:56:09 INFO     [api.services.search.context] restructure_context_cached: hit (alias group)
2026-03-19 10:56:09 INFO     [api.services.search.coverage] build_prompt_result_index: 100 runs -> 15 unique prompts, 3027 total query results


Active steps: ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching']
  Excluded: ['llm_ranking']
  Restructured baseline fields (cached):
    persona: You are a candidate evaluation expert.
    task_intent: Summarize the entity profile, identify its category and key distinguishing featu...
    problem_description: Given an entity_profile_json and a list of candidate matches, produce a concise ...
    instruction: TASK 1: Summarize the profile in 1‑2 sentences, identify entity_category, and li...
    thinking_style: Think step by step.
    answer_format: JSON with keys "reasoning" (string) and "ranked_candidates" (array of objects co...
  Search baseline: 3bb4bbfba4b6 (render: 1154 chars)

  Historical data: 3027 results across 15 unique prompts
  Baseline alias group: 14889901 ↔ 239cefb8 ↔ 24aeb9e7 ↔ 2c14e76c ↔ 2d9d15f6 ↔ 41f88bae ↔ 44eb12a0 ↔ 4ff72b79 ↔ 61ad2b63 ↔ 82f3e7e2 ↔ 830faccd ↔ 9e4f0633 ↔ adb2589d ↔ aeb18154 ↔ bcdc7b72 ↔ c2a36fe9 ↔ c311136d ↔ ce

In [13]:
#@title Sensitivity scan
scan_df, axis_profiles = await sensitivity_scan(
    scan_baseline_sp, scan_variants, eval_data,
    sample_size=scan_sample_size,
    svc=svc, experiment_id=EXPERIMENT_ID or "",
)

Running sensitivity scan...

  Baseline field values:
    persona: You are a candidate evaluation expert.
    task_intent: Summarize the entity profile, identify its category and key distinguishing featu...
    problem_description: Given an entity_profile_json and a list of candidate matches, produce a concise ...
    instruction: TASK 1: Summarize the profile in 1‑2 sentences, identify entity_category, and li...
    thinking_style: Think step by step.
    answer_format: JSON with keys "reasoning" (string) and "ranked_candidates" (array of objects co...

  Axes: 5, variants: 17, queries/variant: 10, cached results: 11563
  Estimated calls: ~170
  Evaluating baseline...
        MISS 2/20  [token] 📖  PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Glass fibre reinforced plastic | 95 4.0s
        HIT   [token] 📖  Stainless steel EN 10270-3/winding             -> Wire drawing, steel {RER}| wire dra 4.1s
        MISS 5/20  [token] 📖  SJRG0010-ABS/molding                           -> Acryl

In [ ]:
# #@title Scan analytics (uncomment to display)
# if scan_df is not None and not scan_df.empty:
#     show_scan_leaderboard(scan_df, axis_profiles)
#     difficulty_df = show_scan_query_difficulty(svc["store"], svc["backend_id"])

In [14]:
#@title Select scan winner & seed campaign
best_sp = seed_campaign_from_scan(
    scan_df, axis_profiles, scan_baseline_sp, scan_variants,
    campaign_rounds, campaign_config,
)

2026-03-19 10:56:16 INFO     [api.services.search.scan_winner] select_scan_winner: 0 prompt changes, 4 param changes from 4 improving axes


Selected best from 4 improving axes:
  max_token_candidates      best_delta=+18.0%  value_idx=1  acc=30.0%
  query_prefix              best_delta=+9.2%  value_idx=0  acc=20.0%
  profiling_schema          best_delta=+9.2%  value_idx=2  acc=20.0%
  profiling_temperature     best_delta=+9.0%  value_idx=1  acc=20.0%
Pipeline params updated: {'steps': ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching'], 'max_token_candidates': 30, 'query_prefix': 'what material is', 'profiling_schema': {'type': 'object', 'properties': {'entity_name': {'type': 'string'}, 'core_concept': {'type': 'string', 'description': 'The single word that defines what this expression represents'}, 'distinguishing_features': {'type': 'array', 'items': {'type': 'string'}}, 'key_properties': {'type': 'array', 'items': {'type': 'string'}}, 'technical_specifications': {'type': 'array', 'items': {'type': 'string'}, 'description': 'Explicit technical specs, dimensions, codes, ratings, tolerance

### 3b. Grid Search

<details>
<summary>Grid search cells (click to expand)</summary>

Systematic sweep of the prompt configuration space. Maps the accuracy landscape before hill-climbing. All cells below are commented out by default.

**To activate:** uncomment cells below and run in order. Grid search evaluates all combinations of prompt fields and pipeline params — expect 100–500+ backend calls depending on `grid_budget` and `sample_size` in `campaign_config["grid_search"]`.

</details>

In [ ]:
# #@title Grid campaign overview (existing plans)
# merge_plans = False  # Set True to combine results from multiple plans
# grid_overview = show_grid_overview(svc, campaign_config, merge_plans=merge_plans)
# merged_grid_df = grid_overview.get("merged_grid_df")

In [ ]:
# #@title Build or resume grid plan
# gs = campaign_config["grid_search"]

# llm_client, llm_model = setup_llm(campaign_config)

# (
#     grid_plan_id, grid_points, grid_state_lookup,
#     grid_axes, layer1_fields, grid_baseline,
# ) = await resume_or_build_grid(
#     campaign_config, baseline, llm_client, llm_model,
#     svc["store"], svc["backend_id"],
#     improvement_areas=campaign_config.get("improvement_areas", ""),
# )

# print(f"Grid points: {len(grid_points)}")
# print(f"Plan ID: {grid_plan_id}")

In [ ]:
# #@title Run grid search
# grid_df = await run_grid_search(
#     grid_points, grid_state_lookup, eval_data,
#     campaign_config["eval_llm"],
#     plan_id=grid_plan_id,
#     svc=svc,
#     pipeline_params=campaign_config.get("pipeline_params"),
#     sample_size=gs.get("sample_size", 1),
#     shared_queries=gs.get("shared_queries", False),
#     grid_seed=gs.get("seed", 42),
# )

In [ ]:
# #@title Display grid results
# _display_df = merged_grid_df if merged_grid_df is not None else grid_df
# display_grid_results(_display_df, grid_axes, top_k=gs.get("top_k", 5))

In [ ]:
# #@title LLM analysis of grid results
# _analysis_df = merged_grid_df if merged_grid_df is not None else grid_df
# llm_client, llm_model = setup_llm(campaign_config)
# grid_analysis = await analyze_grid_results(
#     _analysis_df, grid_axes, llm_client, model=llm_model,
# )

In [ ]:
# #@title Select grid winner and seed campaign
# grid_winner = select_and_seed_grid_winner(
#     grid_df, merged_grid_df, grid_state_lookup,
#     grid_overview.get("plan_dfs", {}), svc, campaign_rounds,
# )

## 4. Optimize

Two modes: **Semi-automatic** (feedback cycle with patience-based auto-stop) or **Manual** (one round at a time).

In [15]:
#@title Feedback cycle preflight
scan_context = show_feedback_preflight(
    campaign_rounds, eval_data, campaign_config,
    pipeline_params=pipeline_params,
    scan_df=locals().get("scan_df"),
    axis_profiles=locals().get("axis_profiles"),
    scan_variants=locals().get("scan_variants"),
    difficulty_df=locals().get("difficulty_df"),
)


  FEEDBACK CYCLE PRE-FLIGHT
  Baseline accuracy      : 10.0%
  Baseline prompt        : TASK 1: Summarize the profile in 1‑2 sentences, identify entity_category, and li...
  ------------------------------------------------------------------
  Max rounds             : 3
  Candidates per round   : 5
  Queries per eval       : 15 of 984
  Improvement threshold  : 1.0%
  Patience (L1)          : 2 rounds
  L2 (refine context)    : enabled, patience=None
  L3 (modify plan)       : enabled, patience=None
  ------------------------------------------------------------------
  Candidate model        : openai/gpt-oss-120b
  Creativity             : 0.7
  Pipeline               : 5 of 6 steps
    Steps                : cache_lookup, fuzzy_matching, web_search, entity_profiling, token_matching
    Excluded             : llm_ranking
  Strategy               : SCAN-AWARE

  ROUND PIPELINE (what happens each round)
  ------------------------------------------------------------------
  1. BASELINE IN

In [ ]:
#@title Run optimization (feedback cycle)
# Force-reload api modules (ensures code edits take effect without kernel restart)
import importlib, sys
for _m in [
    "api.services.campaign.escalation",
    "api.services.campaign.layer_transitions",
    "api.services.campaign.critique",
    "api.services.campaign.models",
    "api.services.prompt_optimizer",
    "api.nodes.optimizer_nodes",
    "api.services.campaign.feedback_cycle",
]:
    if _m in sys.modules:
        importlib.reload(sys.modules[_m])

campaign_rounds = await run_feedback_cycle_notebook(
    campaign_rounds, eval_data, campaign_config,
    svc=svc,
    pipeline_params=pipeline_params,
    scan_context=locals().get("scan_context"),
    experiment_id=locals().get("EXPERIMENT_ID"),
    task_context=locals().get("task_context"),
)

In [ ]:
#@title 5. Results — Campaign comparison, flip tracking, lineage
show_campaign_summary(campaign_rounds)
show_flip_tracking(campaign_rounds)
show_lineage_chain(campaign_rounds)

In [ ]:
#@title Save winner
save_campaign_winner(
    campaign_rounds, campaign_config, svc["store"], svc["backend_id"],
    experiment_id=locals().get("EXPERIMENT_ID"),
)

In [ ]:
#@title Generate LLM suggestions for next round
llm_client, llm_model = setup_llm(campaign_config)
suggestions = await generate_suggestions(
    campaign_rounds, eval_data, campaign_config,
    llm_client, model=llm_model,
)
display_suggestions(suggestions, len(campaign_rounds))
print("--- SUGGESTED CONFIG (copy to Setup) ---")
print(json.dumps(suggestions.get("suggested_config", campaign_config), indent=2))

In [ ]:
#@title Sync evaluation history to Langfuse
# Safe to re-run — already-pushed runs are skipped automatically.
stats = sync_langfuse(
    svc["store"], svc["backend_id"],
    dataset_name="termnorm_ground_truth",
)